# Extract Experience Years from Requirements Text

**Purpose:** Extract years of experience from `requirements_text` and populate `experience_years_min` and `experience_years_max` fields

**Process:**
1. Load jobs with requirements_text from database
2. Extract experience years using pattern matching
3. Parse minimum and maximum years required
4. Update database with extracted values

**Run cells in order**

## 1. Install Dependencies

In [13]:
!pip install pymysql sqlalchemy pandas tqdm cryptography

## 2. Import Libraries

In [14]:
import pymysql
import pandas as pd
import json
import re
import ssl
import tempfile
import os
from sqlalchemy import create_engine, text
from tqdm.notebook import tqdm
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries imported successfully')

✅ Libraries imported successfully


## 3. Database Configuration

In [15]:
# ========== DATABASE CONFIG ==========
DB_CONFIG = {
    'host': 'gateway01.ap-southeast-1.prod.aws.tidbcloud.com',
    'port': 4000,
    'user': '4GJhpnEevqoZfyD.root',
    'password': 'oiK2dgnVVJLVHL4v',
    'database': 'data-mining',
    'charset': 'utf8mb4'
}

# ========== SSL CERTIFICATE ==========
CA_CERT_CONTENT = """
-----BEGIN CERTIFICATE-----
MIIFazCCA1OgAwIBAgIRAIIQz7DSQONZRGPgu2OCiwAwDQYJKoZIhvcNAQELBQAw
TzELMAkGA1UEBhMCVVMxKTAnBgNVBAoTIEludGVybmV0IFNlY3VyaXR5IFJlc2Vh
cmNoIEdyb3VwMRUwEwYDVQQDEwxJU1JHIFJvb3QgWDEwHhcNMTUwNjA0MTEwNDM4
WhcNMzUwNjA0MTEwNDM4WjBPMQswCQYDVQQGEwJVUzEpMCcGA1UEChMgSW50ZXJu
ZXQgU2VjdXJpdHkgUmVzZWFyY2ggR3JvdXAxFTATBgNVBAMTDElTUkcgUm9vdCBY
MTCCAiIwDQYJKoZIhvcNAQEBBQADggIPADCCAgoCggIBAK3oJHP0FDfzm54rVygc
h77ct984kIxuPOZXoHj3dcKi/vVqbvYATyjb3miGbESTtrFj/RQSa78f0uoxmyF+
0TM8ukj13Xnfs7j/EvEhmkvBioZxaUpmZmyPfjxwv60pIgbz5MDmgK7iS4+3mX6U
A5/TR5d8mUgjU+g4rk8Kb4Mu0UlXjIB0ttov0DiNewNwIRt18jA8+o+u3dpjq+sW
T8KOEUt+zwvo/7V3LvSye0rgTBIlDHCNAymg4VMk7BPZ7hm/ELNKjD+Jo2FR3qyH
B5T0Y3HsLuJvW5iB4YlcNHlsdu87kGJ55tukmi8mxdAQ4Q7e2RCOFvu396j3x+UC
B5iPNgiV5+I3lg02dZ77DnKxHZu8A/lJBdiB3QW0KtZB6awBdpUKD9jf1b0SHzUv
KBds0pjBqAlkd25HN7rOrFleaJ1/ctaJxQZBKT5ZPt0m9STJEadao0xAH0ahmbWn
OlFuhjuefXKnEgV4We0+UXgVCwOPjdAvBbI+e0ocS3MFEvzG6uBQE3xDk3SzynTn
jh8BCNAw1FtxNrQHusEwMFxIt4I7mKZ9YIqioymCzLq9gwQbooMDQaHWBfEbwrbw
qHyGO0aoSCqI3Haadr8faqU9GY/rOPNk3sgrDQoo//fb4hVC1CLQJ13hef4Y53CI
rU7m2Ys6xt0nUW7/vGT1M0NPAgMBAAGjQjBAMA4GA1UdDwEB/wQEAwIBBjAPBgNV
HRMBAf8EBTADAQH/MB0GA1UdDgQWBBR5tFnme7bl5AFzgAiIyBpY9umbbjANBgkq
hkiG9w0BAQsFAAOCAgEAVR9YqbyyqFDQDLHYGmkgJykIrGF1XIpu+ILlaS/V9lZL
ubhzEFnTIZd+50xx+7LSYK05qAvqFyFWhfFQDlnrzuBZ6brJFe+GnY+EgPbk6ZGQ
3BebYhtF8GaV0nxvwuo77x/Py9auJ/GpsMiu/X1+mvoiBOv/2X/qkSsisRcOj/KK
NFtY2PwByVS5uCbMiogziUwthDyC3+6WVwW6LLv3xLfHTjuCvjHIInNzktHCgKQ5
ORAzI4JMPJ+GslWYHb4phowim57iaztXOoJwTdwJx4nLCgdNbOhdjsnvzqvHu7Ur
TkXWStAmzOVyyghqpZXjFaH3pO3JLF+l+/+sKAIuvtd7u+Nxe5AW0wdeRlN8NwdC
jNPElpzVmbUq4JUagEiuTDkHzsxHpFKVK7q4+63SM1N95R1NbdWhscdCb+ZAJzVc
oyi3B43njTOQ5yOf+1CceWxG1bQVs5ZufpsMljq4Ui0/1lvh+wjChP4kqKOJ2qxq
4RgqsahDYVvTH9w7jXbyLeiNdd8XM2w9U/t7y0Ff/9yi0GE44Za4rF2LN9d11TPA
mRGunUHBcnWEvgJBQl9nJEiU0Zsnvgc/ubhPgXRR4Xq37Z0j4r7g1SgEEzwxA57d
emyPxgcYxn/eR44/KJ4EBs+lVDR3veyJm+kXQ99b21/+jh5Xos1AnX5iItreGCc=
-----END CERTIFICATE-----
""".strip()

print(f'📊 Database: {DB_CONFIG["database"]}')
print(f'🔗 Host: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')
print(f'📜 SSL configured: ✅')

📊 Database: data-mining
🔗 Host: gateway01.ap-southeast-1.prod.aws.tidbcloud.com:4000
📜 SSL configured: ✅


## 4. Setup Database Connection

In [16]:
temp_files = []
try:
    ssl_context = ssl.create_default_context()
    
    if CA_CERT_CONTENT:
        ca_temp = tempfile.NamedTemporaryFile(mode='w', suffix='.pem', delete=False)
        ca_temp.write(CA_CERT_CONTENT)
        ca_temp.close()
        temp_files.append(ca_temp.name)
        ssl_context.load_verify_locations(ca_temp.name)
        print('✅ Loaded CA certificate')
    
    DATABASE_URL = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}?charset={DB_CONFIG['charset']}"
    engine = create_engine(DATABASE_URL, connect_args={'ssl': ssl_context}, pool_pre_ping=True)
    
    print('\n🔄 Testing connection...')
    with engine.connect() as conn:
        result = conn.execute(text('SELECT COUNT(*) FROM jobs'))
        total = result.fetchone()[0]
        print(f'\n✅ Connection successful!')
        print(f'📊 Total jobs: {total:,}')
        
except Exception as e:
    print(f'❌ Error: {e}')
    for f in temp_files:
        try: os.unlink(f)
        except: pass
    raise

✅ Loaded CA certificate

🔄 Testing connection...

✅ Connection successful!
📊 Total jobs: 12,117


## 5. Load Data from Database

In [17]:
print('📥 Loading jobs data...')
query = '''
SELECT 
    id, 
    title,
    requirements_text,
    description,
    experience_years_min,
    experience_years_max,
    company_name,
    level
FROM jobs 
WHERE (requirements_text IS NOT NULL AND requirements_text != '')
   OR (description IS NOT NULL AND description != '')
ORDER BY id
'''

df = pd.read_sql(query, engine)
print(f'✅ Loaded {len(df):,} jobs')

# Show current state
has_min = df['experience_years_min'].notna().sum()
has_max = df['experience_years_max'].notna().sum()
has_both = ((df['experience_years_min'].notna()) & (df['experience_years_max'].notna())).sum()

print(f'\n📊 Current state:')
print(f'  experience_years_min filled: {has_min:,} ({has_min/len(df)*100:.1f}%)')
print(f'  experience_years_max filled: {has_max:,} ({has_max/len(df)*100:.1f}%)')
print(f'  Both fields filled: {has_both:,} ({has_both/len(df)*100:.1f}%)')

df.head(3)

📥 Loading jobs data...
✅ Loaded 11,989 jobs

📊 Current state:
  experience_years_min filled: 9,859 (82.2%)
  experience_years_max filled: 9,859 (82.2%)
  Both fields filled: 9,859 (82.2%)


,id,title,requirements_text,description,experience_years_min,experience_years_max,company_name,level
0,3462,Frontend developer (PA project),Your skills & qualifications:\n\nExperience in...,Your role & responsibilities:\nWe’re looking f...,1.0,1.0,CUBICASA,Manager
1,3470,TIGER TRIBE – SENIOR BACKEND DEVELOPER,Your skills & qualifications:\n\n5+ years of e...,"Your role & responsibilities:\nDesign, develop...",5.0,5.0,TIGER TRIBE,Senior
2,3474,Frontend developer (PA project),Your skills & qualifications:\n\nExperience in...,Your role & responsibilities:\nWe’re looking f...,1.0,1.0,CUBICASA,Manager


## 6. Define Experience Extraction Class

In [18]:
class ExperienceExtractor:
    """
    Extract years of experience from job requirements text
    Optimized for English language patterns
    """
    
    def __init__(self):
        # Compile regex patterns for experience extraction
        self.patterns = [
            # "X+ years" or "X years+" or "X plus years"
            r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:of\s*)?(?:experience|exp)?',
            r'(\d+)\s*(?:years?|yrs?)\+',
            r'(\d+)\s*plus\s*(?:years?|yrs?)',
            
            # "X to Y years" or "X-Y years" (various separators)
            r'(\d+)\s*(?:to|-|~|–|—|through|thru)\s*(\d+)\s*(?:years?|yrs?)',
            
            # "At least X years" or "minimum X years"
            r'(?:at\s*least|minimum|min\.?|from|starting\s+from)\s*(\d+)\s*(?:years?|yrs?)',
            
            # "X years of experience/expertise"
            r'(\d+)\+?\s*(?:years?|yrs?)\s*(?:of\s*)?(?:experience|exp|expertise|background)',
            
            # "Experience: X years" or "Experience: X+ years"
            r'(?:experience|exp)[:\s]+(\d+)\+?\s*(?:years?|yrs?)',
            
            # "with X years" or "have X years" or "possess X years"
            r'(?:with|have|has|possess|possessing)\s+(\d+)\+?\s*(?:years?|yrs?)',
            
            # "requires X years" or "need X years" or "seeking X years"
            r'(?:requires?|needs?|must\s+have|seeking|looking\s+for|prefer)\s+(\d+)\+?\s*(?:years?|yrs?)',
            
            # "X+ year" (singular)
            r'(\d+)\+\s*year(?!s)',
            
            # "Over X years" or "more than X years" or "above X years"
            r'(?:over|more\s+than|above|exceeding|upwards?\s+of)\s+(\d+)\s*(?:years?|yrs?)',
            
            # "Between X and Y years"
            r'between\s+(\d+)\s+and\s+(\d+)\s*(?:years?|yrs?)',
            
            # "X or more years" or "X and above"
            r'(\d+)\s*(?:or\s+)?(?:more|above)\s*(?:years?|yrs?)',
            
            # "Up to X years" (maximum)
            r'(?:up\s+to|maximum|max\.?|not\s+more\s+than)\s*(\d+)\s*(?:years?|yrs?)',
            
            # "Less than X years" (for junior positions)
            r'(?:less\s+than|under|below)\s*(\d+)\s*(?:years?|yrs?)',
            
            # "X years in [technology/field]"
            r'(\d+)\+?\s*(?:years?|yrs?)\s+(?:in|of|with|using)',
            
            # "0-1 years" or "1-2 years" (common formats)
            r'(\d+)\s*-\s*(\d+)\s*(?:years?|yrs?)',
            
            # "Prefer X years" or "Ideally X years"
            r'(?:prefer|preferred|ideally|desirable)\s+(\d+)\+?\s*(?:years?|yrs?)',
            
            # "X years' experience" (with apostrophe)
            r'(\d+)\s*(?:years?|yrs?)[\'\']\s*(?:experience|exp)',
            
            # Entry level indicators (0 years)
            r'(?:entry[\s-]?level|junior|graduate|fresh|no[\s-]?experience|fresher)',
            
            # "0 years" or "0-1 years" explicitly
            r'\b0\s*(?:years?|yrs?)',
            r'\b0\s*-\s*(\d+)\s*(?:years?|yrs?)',
            
            # Written numbers - "one year", "two years", etc.
            r'\b(one|two|three|four|five|six|seven|eight|nine|ten)\s*(?:years?|yrs?)',
            
            # Written number ranges - "one to three years", "two to five years"
            r'\b(one|two|three|four|five)\s+(?:to|-)\s+(two|three|four|five|six|seven|eight|nine|ten)\s*(?:years?|yrs?)',
        ]
        
        self.compiled_patterns = [re.compile(p, re.IGNORECASE) for p in self.patterns]
        
        # Mapping for written numbers to integers
        self.word_to_num = {
            'zero': 0, 'one': 1, 'two': 2, 'three': 3, 'four': 4,
            'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10
        }
        
        self.stats = {
            'processed': 0,
            'found_min': 0,
            'found_max': 0,
            'found_both': 0,
            'not_found': 0,
            'from_requirements': 0,
            'from_description': 0,
            'from_level': 0
        }
    
    def convert_word_to_number(self, word):
        """Convert written number to integer"""
        return self.word_to_num.get(word.lower(), None)
    
    def extract_from_level(self, level):
        """
        Infer experience years from level field as fallback
        Returns: (min_years, max_years) or (None, None)
        """
        if not level or not str(level).strip():
            return None, None
        
        level = str(level).lower().strip()
        
        # Entry level / Junior / Fresher (0-2 years)
        if re.search(r'\b(entry|junior|jr\.?|fresher|intern|graduate|trainee)\b', level):
            return 0, 2
        
        # Mid level / Middle (2-5 years)
        if re.search(r'\b(mid|middle|intermediate)\b', level):
            return 2, 5
        
        # Senior (5-10 years)
        if re.search(r'\b(senior|sr\.?)\b', level):
            return 5, 10
        
        # Lead (5-10 years)
        if re.search(r'\b(lead|team\s*lead)\b', level):
            return 5, 10
        
        # Principal / Staff / Expert (8-15 years)
        if re.search(r'\b(principal|staff|expert|specialist)\b', level):
            return 8, 15
        
        # Manager (5-15 years - wide range as it varies)
        if re.search(r'\b(manager|mgr)\b', level):
            return 5, 15
        
        # Director / Head (10-20 years)
        if re.search(r'\b(director|head\s+of)\b', level):
            return 10, 20
        
        # C-Level / VP (15-25 years)
        if re.search(r'\b(vp|vice\s*president|cto|ceo|cio|cfo)\b', level):
            return 15, 25
        
        # If level has numeric value (e.g., "Level 3", "L3")
        level_num_match = re.search(r'\b(?:level|l)\s*(\d+)\b', level)
        if level_num_match:
            level_num = int(level_num_match.group(1))
            # Rough mapping: Level 1-2 -> 0-2 years, Level 3-4 -> 2-5 years, Level 5+ -> 5-10 years
            if level_num <= 2:
                return 0, 2
            elif level_num <= 4:
                return 2, 5
            else:
                return 5, 10
        
        return None, None
    
    def extract_from_text(self, text):
        """
        Extract years of experience from text using regex patterns
        Returns: (min_years, max_years) or (None, None)
        """
        if not text or not str(text).strip():
            return None, None
        
        text = str(text).lower()
        
        # Check for entry level indicators first
        entry_pattern = r'\b(entry[\s-]?level|junior|graduate|fresh|no[\s-]?experience|fresher)\b'
        if re.search(entry_pattern, text):
            return 0, 2
        
        all_years = []
        
        for pattern in self.compiled_patterns:
            matches = pattern.finditer(text)
            for match in matches:
                groups = match.groups()
                
                # Filter out false positives (e.g., "in 5 years" means future, not experience)
                match_text = match.group(0)
                if re.search(r'\b(?:in|within|for\s+the\s+next|after)\s+\d+\s*years?\b', match_text):
                    continue  # Skip false positives
                
                # Single number match
                if len(groups) == 1 and groups[0]:
                    try:
                        # Check if it's a written number
                        years = self.convert_word_to_number(groups[0])
                        if years is None:
                            years = int(groups[0])
                        all_years.append((years, years))
                    except:
                        pass
                
                # Range match (X to Y)
                elif len(groups) >= 2 and groups[0] and groups[1]:
                    try:
                        # Handle written numbers in ranges
                        min_years = self.convert_word_to_number(groups[0])
                        if min_years is None:
                            min_years = int(groups[0])
                        
                        max_years = self.convert_word_to_number(groups[1])
                        if max_years is None:
                            max_years = int(groups[1])
                        
                        all_years.append((min_years, max_years))
                    except:
                        pass
        
        if not all_years:
            return None, None
        
        # If multiple matches found, use the most conservative (smallest min, largest max)
        min_years = min(y[0] for y in all_years)
        max_years = max(y[1] for y in all_years)
        
        # Validate: max should be >= min
        if max_years < min_years:
            max_years = min_years
        
        # Handle "less than X" patterns - these indicate maximum, not minimum
        less_than_pattern = r'(?:less\s+than|under|below)\s*(\d+)\s*(?:years?|yrs?)'
        less_than_match = re.search(less_than_pattern, text, re.IGNORECASE)
        if less_than_match and not re.search(r'(?:at\s*least|minimum|from)', text, re.IGNORECASE):
            # This is a maximum requirement, not minimum
            try:
                max_val = int(less_than_match.group(1))
                return 0, max_val
            except:
                pass
        
        # Cap at reasonable maximum (30 years)
        if max_years > 30:
            max_years = 30
        if min_years > 30:
            min_years = 30
        
        # If min and max are both 0, interpret as 0-2 for entry level
        if min_years == 0 and max_years == 0:
            max_years = 2
        
        return min_years, max_years
    
    def extract(self, row):
        """
        Extract experience years from job data with priority:
        1. requirements_text (most explicit)
        2. description
        3. level (fallback inference)
        """
        requirements = row.get('requirements_text', '')
        description = row.get('description', '')
        level = row.get('level', '')
        
        self.stats['processed'] += 1
        
        # Priority 1: Check requirements_text
        if requirements and str(requirements).strip():
            min_years, max_years = self.extract_from_text(requirements)
            if min_years is not None:
                self.stats['from_requirements'] += 1
                self._update_stats(min_years, max_years)
                return min_years, max_years
        
        # Priority 2: Check description
        if description and str(description).strip():
            min_years, max_years = self.extract_from_text(description)
            if min_years is not None:
                self.stats['from_description'] += 1
                self._update_stats(min_years, max_years)
                return min_years, max_years
        
        # Priority 3: Infer from level field (fallback)
        if level and str(level).strip():
            min_years, max_years = self.extract_from_level(level)
            if min_years is not None:
                self.stats['from_level'] += 1
                self._update_stats(min_years, max_years)
                return min_years, max_years
        
        # No explicit experience requirement found
        self.stats['not_found'] += 1
        return None, None
    
    def _update_stats(self, min_years, max_years):
        """Update extraction statistics"""
        if min_years is not None:
            self.stats['found_min'] += 1
        if max_years is not None:
            self.stats['found_max'] += 1
        if min_years is not None and max_years is not None:
            self.stats['found_both'] += 1
    
    def print_stats(self):
        """Print extraction statistics"""
        print(f"📊 EXTRACTION STATISTICS:")
        print(f"  Processed jobs: {self.stats['processed']:,}")
        print(f"  Found min years: {self.stats['found_min']:,} ({self.stats['found_min']/self.stats['processed']*100:.1f}%)")
        print(f"  Found max years: {self.stats['found_max']:,} ({self.stats['found_max']/self.stats['processed']*100:.1f}%)")
        print(f"  Found both: {self.stats['found_both']:,} ({self.stats['found_both']/self.stats['processed']*100:.1f}%)")
        print(f"  Not found: {self.stats['not_found']:,} ({self.stats['not_found']/self.stats['processed']*100:.1f}%)")
        print(f"\n  Source breakdown:")
        print(f"    From requirements: {self.stats['from_requirements']:,}")
        print(f"    From description: {self.stats['from_description']:,}")
        print(f"    From level: {self.stats['from_level']:,}")

print('✅ ExperienceExtractor class defined')
print('🔍 Extraction method: Advanced regex pattern matching')
print('📋 Priority: requirements_text > description > level')
print('🎯 Level fallback: Uses job level to infer experience when not explicitly stated')
print('🌐 Language: English only (optimized)')
print('✨ Features: 25+ patterns, written numbers, false positive filtering')

✅ ExperienceExtractor class defined
🔍 Extraction method: Advanced regex pattern matching
📋 Priority: requirements_text > description > level
🎯 Level fallback: Uses job level to infer experience when not explicitly stated
🌐 Language: English only (optimized)
✨ Features: 25+ patterns, written numbers, false positive filtering


## 7. Extract Experience Years from All Jobs

In [19]:
import time

extractor = ExperienceExtractor()
df_updated = df.copy()

print(f'🚀 Starting experience extraction for {len(df):,} jobs')
print(f'🔥 Mode: FULL EXTRACTION - Will update ALL jobs')
print(f'⏱️  Estimated time: ~{len(df)/1000:.1f} seconds\n')

start_time = time.time()
updated_count = 0

for idx in tqdm(range(len(df_updated)), desc='Extracting experience years'):
    row = df_updated.iloc[idx]
    
    # Extract experience years
    min_years, max_years = extractor.extract(row)
    
    # Update fields
    if min_years is not None:
        df_updated.at[idx, 'experience_years_min'] = min_years
        updated_count += 1
    
    if max_years is not None:
        df_updated.at[idx, 'experience_years_max'] = max_years
    
    # Progress reporting every 1000 jobs
    if (idx + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        rate = (idx + 1) / elapsed
        remaining = (len(df_updated) - idx - 1) / rate
        print(f'\n[{idx+1}/{len(df_updated)}] Time: {elapsed:.1f}s | '
              f'Remaining: {remaining:.1f}s | Rate: {rate:.1f} jobs/s')
        extractor.print_stats()

elapsed_time = time.time() - start_time
print(f'\n✅ Extraction complete!')
print(f'⏱️  Total time: {elapsed_time:.1f} seconds')
print(f'⚡ Average rate: {len(df)/elapsed_time:.1f} jobs/second')
print(f'\n📊 FINAL STATISTICS:')


extractor.print_stats()
print(f'📝 Jobs with experience extracted: {updated_count:,} ({updated_count/len(df)*100:.1f}%)')

🚀 Starting experience extraction for 11,989 jobs
🔥 Mode: FULL EXTRACTION - Will update ALL jobs
⏱️  Estimated time: ~12.0 seconds



Extracting experience years:   0%|          | 0/11989 [00:00<?, ?it/s]


[1000/11989] Time: 1.7s | Remaining: 18.9s | Rate: 580.8 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 1,000
  Found min years: 985 (98.5%)
  Found max years: 985 (98.5%)
  Found both: 985 (98.5%)
  Not found: 15 (1.5%)

  Source breakdown:
    From requirements: 821
    From description: 46
    From level: 118

[2000/11989] Time: 2.9s | Remaining: 14.4s | Rate: 691.6 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 2,000
  Found min years: 1,965 (98.2%)
  Found max years: 1,965 (98.2%)
  Found both: 1,965 (98.2%)
  Not found: 35 (1.8%)

  Source breakdown:
    From requirements: 1,609
    From description: 57
    From level: 299

[3000/11989] Time: 4.4s | Remaining: 13.3s | Rate: 674.9 jobs/s
📊 EXTRACTION STATISTICS:
  Processed jobs: 3,000
  Found min years: 2,947 (98.2%)
  Found max years: 2,947 (98.2%)
  Found both: 2,947 (98.2%)
  Not found: 53 (1.8%)

  Source breakdown:
    From requirements: 2,388
    From description: 84
    From level: 475

[4000/11989] Time: 10.4s | Re

## 8. Preview Extracted Experience

In [20]:
print('🔍 SAMPLE EXPERIENCE EXTRACTIONS (First 20 jobs with changes)\n')
print('='*100)

shown = 0
for idx in range(len(df_updated)):
    old_row = df.iloc[idx]
    new_row = df_updated.iloc[idx]
    
    # Check if values changed
    min_changed = str(old_row['experience_years_min']) != str(new_row['experience_years_min'])
    max_changed = str(old_row['experience_years_max']) != str(new_row['experience_years_max'])
    
    if min_changed or max_changed:
        print(f"\n📋 Job ID: {new_row['id']}")
        print(f"   Title: {new_row['title']}")
        print(f"   Company: {new_row['company_name']}")
        print(f"   Level: {new_row['level']}")
        
        if min_changed:
            old_val = old_row['experience_years_min'] if pd.notna(old_row['experience_years_min']) else 'NULL'
            new_val = new_row['experience_years_min'] if pd.notna(new_row['experience_years_min']) else 'NULL'
            print(f"   ✅ Min years: {old_val} → {new_val}")
        
        if max_changed:
            old_val = old_row['experience_years_max'] if pd.notna(old_row['experience_years_max']) else 'NULL'
            new_val = new_row['experience_years_max'] if pd.notna(new_row['experience_years_max']) else 'NULL'
            print(f"   ✅ Max years: {old_val} → {new_val}")
        
        # Show excerpt from requirements
        if pd.notna(new_row['requirements_text']):
            req_excerpt = str(new_row['requirements_text'])[:150]
            print(f"   📄 Requirements excerpt: {req_excerpt}...")
        
        shown += 1
        if shown >= 20:
            break

print('\n' + '='*100)

# Show distribution
print('\n📊 EXPERIENCE YEARS DISTRIBUTION:\n')

min_counts = df_updated['experience_years_min'].value_counts().sort_index()
print('Min Years Required:')
for years, count in min_counts.head(10).items():
    if pd.notna(years):
        percentage = count / len(df) * 100
        print(f"  {int(years)} years: {count:,} jobs ({percentage:.1f}%)")

print('\nMost Common Experience Ranges:')
df_updated['exp_range'] = df_updated.apply(
    lambda x: f"{int(x['experience_years_min'])}-{int(x['experience_years_max'])} years" 
    if pd.notna(x['experience_years_min']) and pd.notna(x['experience_years_max']) 
    else None, 
    axis=1
)
range_counts = df_updated['exp_range'].value_counts().head(10)
for exp_range, count in range_counts.items():
    if exp_range:
        percentage = count / len(df) * 100
        print(f"  {exp_range}: {count:,} jobs ({percentage:.1f}%)")

🔍 SAMPLE EXPERIENCE EXTRACTIONS (First 20 jobs with changes)


📋 Job ID: 3462
   Title: Frontend developer (PA project)
   Company: CUBICASA
   Level: Manager
   ✅ Min years: 1.0 → 5.0
   ✅ Max years: 1.0 → 15.0
   📄 Requirements excerpt: Your skills & qualifications:

Experience in front end development with HTML, CSS, Sass, Typescript 


Knowledge of modern JS frameworks, preferably V...

📋 Job ID: 3474
   Title: Frontend developer (PA project)
   Company: CUBICASA
   Level: Manager
   ✅ Min years: 1.0 → 5.0
   ✅ Max years: 1.0 → 15.0
   📄 Requirements excerpt: Your skills & qualifications:

Experience in front end development with HTML, CSS, Sass, Typescript 


Knowledge of modern JS frameworks, preferably V...

📋 Job ID: 3475
   Title: Frontend developer (PA project)
   Company: CUBICASA
   Level: Manager
   ✅ Min years: 1.0 → 5.0
   ✅ Max years: 1.0 → 15.0
   📄 Requirements excerpt: Your skills & qualifications:

Experience in front end development with HTML, CSS, Sass, Typescript

## 9. Save Backup CSV

In [21]:
from datetime import datetime

backup_file = f'jobs_experience_extracted_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
df_updated.to_csv(backup_file, index=False, encoding='utf-8-sig')
print(f'✅ Backup saved: {backup_file}')
print(f'📦 File size: {os.path.getsize(backup_file) / 1024 / 1024:.2f} MB')

try:
    from google.colab import files
    files.download(backup_file)
    print('📥 File downloaded')
except:
    print('💾 File saved locally')

✅ Backup saved: jobs_experience_extracted_20260122_084006.csv
📦 File size: 65.47 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 File downloaded


## 10. Update Database

In [22]:
confirm = input('⚠️  Update database with extracted experience years? (yes/no): ')

if confirm.lower() == 'yes':
    print('\n🔄 Updating database...')
    print('⏰ This will update the "updated_at" timestamp for all modified rows\n')
    
    update_count = 0
    error_count = 0
    
    with engine.begin() as conn:
        for idx in tqdm(range(len(df_updated)), desc='Updating database'):
            old_row = df.iloc[idx]
            new_row = df_updated.iloc[idx]
            
            # Only update if values changed
            min_changed = str(old_row['experience_years_min']) != str(new_row['experience_years_min'])
            max_changed = str(old_row['experience_years_max']) != str(new_row['experience_years_max'])
            
            if min_changed or max_changed:
                try:
                    update_query = '''
                        UPDATE jobs SET
                            experience_years_min = :min_years,
                            experience_years_max = :max_years,
                            updated_at = NOW()
                        WHERE id = :id
                    '''
                    
                    params = {
                        'id': int(new_row['id']),
                        'min_years': int(new_row['experience_years_min']) if pd.notna(new_row['experience_years_min']) else None,
                        'max_years': int(new_row['experience_years_max']) if pd.notna(new_row['experience_years_max']) else None
                    }
                    
                    conn.execute(text(update_query), params)
                    update_count += 1
                    
                except Exception as e:
                    error_count += 1
                    if error_count <= 5:
                        print(f'\n⚠️ Error updating job {new_row["id"]}: {str(e)[:100]}')
    
    print(f'\n✅ Updated: {update_count:,} jobs in database')
    print(f'❌ Errors: {error_count:,}')
    
    if error_count == 0:
        # Verify update
        print(f'\n🔍 Verifying database updates...')
        verify_query = 'SELECT COUNT(*) as updated FROM jobs WHERE updated_at >= DATE_SUB(NOW(), INTERVAL 5 MINUTE)'
        result = pd.read_sql(verify_query, engine)
        recently_updated = result['updated'].iloc[0]
        print(f'  ✅ {recently_updated:,} jobs have updated_at within last 5 minutes')
        
        # Show updated counts
        stats_query = '''
            SELECT 
                COUNT(*) as total,
                SUM(CASE WHEN experience_years_min IS NOT NULL THEN 1 ELSE 0 END) as has_min,
                SUM(CASE WHEN experience_years_max IS NOT NULL THEN 1 ELSE 0 END) as has_max,
                SUM(CASE WHEN experience_years_min IS NOT NULL AND experience_years_max IS NOT NULL THEN 1 ELSE 0 END) as has_both
            FROM jobs
        '''
        stats = pd.read_sql(stats_query, engine).iloc[0]
        
        print(f'\n📊 Database statistics:')
        print(f'  Total jobs: {stats["total"]:,}')
        print(f'  Has min years: {stats["has_min"]:,} ({stats["has_min"]/stats["total"]*100:.1f}%)')
        print(f'  Has max years: {stats["has_max"]:,} ({stats["has_max"]/stats["total"]*100:.1f}%)')
        print(f'  Has both: {stats["has_both"]:,} ({stats["has_both"]/stats["total"]*100:.1f}%)')
        
        print(f'\n✅ SUCCESS: Database updated successfully!')
    else:
        print(f'\n⚠️ WARNING: {error_count} errors occurred during update')
else:
    print('❌ Update cancelled by user')


🔄 Updating database...
⏰ This will update the "updated_at" timestamp for all modified rows



Updating database:   0%|          | 0/11989 [00:00<?, ?it/s]


✅ Updated: 2,833 jobs in database
❌ Errors: 0

🔍 Verifying database updates...
  ✅ 1,298 jobs have updated_at within last 5 minutes

📊 Database statistics:
  Total jobs: 12,117.0
  Has min years: 11,725.0 (96.8%)
  Has max years: 11,725.0 (96.8%)
  Has both: 11,725.0 (96.8%)

✅ SUCCESS: Database updated successfully!


## 11. Cleanup and Summary

In [23]:
# Cleanup temporary SSL files
for f in temp_files:
    try:
        os.unlink(f)
        print(f'🗑️  Deleted temp file: {f}')
    except:
        pass

print('\n' + '='*80)
print('🎉 EXPERIENCE EXTRACTION COMPLETE - FINAL SUMMARY')
print('='*80)
print(f'📊 Total jobs processed: {len(df):,}')
print(f'💾 Backup file: {backup_file}')
print(f'⏱️  Processing time: {elapsed_time:.1f} seconds')
print()

extractor.print_stats()
print()

print('📋 MOST COMMON EXPERIENCE REQUIREMENTS:')
for idx, (exp_range, count) in enumerate(range_counts.head(5).items(), 1):
    if exp_range:
        percentage = count / len(df) * 100
        print(f'  {idx}. {exp_range}: {count:,} jobs ({percentage:.1f}%)')
print()

print('='*80)

🗑️  Deleted temp file: /tmp/tmp23hu261h.pem

🎉 EXPERIENCE EXTRACTION COMPLETE - FINAL SUMMARY
📊 Total jobs processed: 11,989
💾 Backup file: jobs_experience_extracted_20260122_084006.csv
⏱️  Processing time: 54.3 seconds

📊 EXTRACTION STATISTICS:
  Processed jobs: 11,989
  Found min years: 11,572 (96.5%)
  Found max years: 11,572 (96.5%)
  Found both: 11,572 (96.5%)
  Not found: 417 (3.5%)

  Source breakdown:
    From requirements: 6,685
    From description: 1,947
    From level: 2,940

📋 MOST COMMON EXPERIENCE REQUIREMENTS:
  1. 0-2 years: 1,657 jobs (13.8%)
  2. 5-10 years: 1,563 jobs (13.0%)
  3. 5-5 years: 1,216 jobs (10.1%)
  4. 3-3 years: 1,154 jobs (9.6%)
  5. 5-15 years: 990 jobs (8.3%)

